# Python. Лекция 13. Данные из внешнего мира

Лектор: Хайбулин Даниэль

Программа редко работает с данными, придуманными ею самой: их читают из файлов,
получают от других программ, выгружают из таблиц. Лекция о том, как это делать
правильно.

Три вопроса, на которые отвечает материал:

| Вопрос | Инструмент |
|---|---|
| где лежит файл | `pathlib` |
| в каком формате записан | `json`, `csv` |
| как разобрать значения | `datetime` |

Все примеры ниже создают учебные файлы сами, поэтому ноутбук можно запускать
целиком и сколько угодно раз.

## Пути: `pathlib`

Путь — это не строка. Строковый путь приходится склеивать вручную, и склейка
ломается между операционными системами: в macOS и Linux разделитель `/`, в
Windows `\`.

`Path` из модуля `pathlib` — объект, который знает про разделители, умеет
разбираться на части и сам ходит в файловую систему.

In [ ]:
from pathlib import Path

DEMO = Path("demo")
DEMO.mkdir(exist_ok=True)

DEMO

Пути соединяют оператором `/` — он перегружен и читается как разделитель:

In [ ]:
report = DEMO / "отчёты" / "январь.csv"

report

Каталога `отчёты` пока нет, но объект пути создан: `Path` описывает путь, а не
существующий файл. Проверить наличие — отдельный вопрос.

In [ ]:
report.exists(), DEMO.exists()

### Части пути

У пути есть имя, расширение, имя без расширения и родительский каталог:

In [ ]:
report.name, report.suffix, report.stem, report.parent

Замена расширения делается методом, а не работой со строкой — иначе легко
испортить имя, в котором точка встречается не один раз:

In [ ]:
report.with_suffix(".json")

### Чтение и запись

Для небольших файлов есть два метода, заменяющих `open` целиком. Кодировку
указывают всегда — по умолчанию она берётся из настроек системы и на другой
машине окажется другой.

In [ ]:
note = DEMO / "заметка.txt"
note.write_text("первая строка\nвторая строка\n", encoding="utf-8")

note.read_text(encoding="utf-8")

Для больших файлов остаётся `open` и построчное чтение: `read_text` поднимает в
память весь файл целиком.

### Обход каталога

`iterdir` перечисляет содержимое, `glob` отбирает по шаблону, `rglob` — то же
самое рекурсивно:

In [ ]:
(DEMO / "пример.csv").write_text("a,b\n1,2\n", encoding="utf-8")

sorted(p.name for p in DEMO.glob("*.csv"))

### Путь относительно модуля

Относительный путь `Path("demo")` отсчитывается от каталога, **из которого
запущена программа**, а не от того, где лежит файл с кодом. Поэтому программа
работает при запуске из корня проекта и падает при запуске из соседней папки.

Устойчивый способ — считать путь от самого модуля:

```python
DATA = Path(__file__).parent / "demo"
```

> **Типичная ошибка.** `FileNotFoundError` при том, что файл на месте. Почти
> всегда это означает, что путь отсчитался не оттуда, откуда вы думали.
> Печать `Path.cwd()` и `path.resolve()` показывает, где программа искала файл.

In [ ]:
Path.cwd()

## JSON

JSON — текстовый формат обмена данными: так отвечают веб-сервисы, так хранят
настройки. Структурно это словари, списки, строки, числа, `true`/`false` и
`null`.

Модуль `json` переводит в обе стороны. Четыре функции, различающиеся по двум
признакам: направление и источник.

| | строка | файл |
|---|---|---|
| разобрать | `loads` | `load` |
| собрать | `dumps` | `dump` |

Буква `s` в названии означает *string*.

In [ ]:
import json

raw = '{"город": "Москва", "население": 13100000, "столица": true}'
data = json.loads(raw)

data

`true` превратился в `True`, число — в `int`, объект — в словарь. Обратная
операция:

In [ ]:
json.dumps(data)

### Кириллица и читаемость

По умолчанию не-ASCII символы экранируются. Файл остаётся корректным, но
человеком не читается — а его почти всегда открывают глазами:

In [ ]:
print(json.dumps(data))
print(json.dumps(data, ensure_ascii=False))

Для файлов, которые кто-то будет читать, добавляют ещё и отступы:

In [ ]:
print(json.dumps(data, ensure_ascii=False, indent=2))

### Что переводится, а что нет

JSON знает меньше типов, чем Python. Часть значений переводится с потерей:

In [ ]:
json.dumps({"кортеж": (1, 2), "ключ-число": {1: "один"}}, ensure_ascii=False)

Кортеж стал списком, а числовой ключ — строкой: в JSON ключи объектов только
строковые. При обратном разборе кортеж уже не восстановится.

Остальные типы вызывают ошибку:

In [ ]:
from datetime import date

for value in ({1, 2}, date(2026, 9, 1)):
    try:
        json.dumps({"значение": value})
    except TypeError as error:
        print(type(value).__name__, "->", error)

Такие значения приводят к поддерживаемым самостоятельно — например, дату к
строке через `isoformat`, о котором ниже.

### Работа с файлом

`dump` и `load` принимают открытый файл. Кодировка задаётся при открытии:

In [ ]:
settings = DEMO / "настройки.json"

with settings.open("w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=2)

with settings.open(encoding="utf-8") as file:
    print(json.load(file))

> **Типичная ошибка.** `json.load(path)` вместо `json.load(file)`. Функция ждёт
> открытый файл, а не путь; со строкой она попытается вызвать у неё `read` и
> упадёт с `AttributeError`. Для строки существует `loads`.

### Битые данные

Разбор чужих данных обязан быть готов к тому, что они неверны:

In [ ]:
try:
    json.loads('{"город": "Москва",}')
except json.JSONDecodeError as error:
    print(type(error).__name__)
    print(error)

`JSONDecodeError` — наследник `ValueError`, и сообщение указывает позицию
ошибки. Ещё одна особенность: повторяющиеся ключи не считаются ошибкой,
побеждает последний.

In [ ]:
json.loads('{"a": 1, "a": 2}')

## CSV

CSV — таблица в текстовом виде: строка файла есть строка таблицы, поля разделены
запятыми. Формат кажется настолько простым, что его постоянно пытаются разобрать
через `split(",")` — и напрасно.

In [ ]:
line = 'Иванов,"Москва, Россия",30'

line.split(",")

Поле, внутри которого есть запятая, берётся в кавычки — и `split` его разрывает.
Правильный разбор делает модуль `csv`, который понимает кавычки, экранирование и
переводы строк внутри полей.

In [ ]:
import csv
import io

next(csv.reader(io.StringIO(line)))

### Чтение файла

Файл открывают с `newline=""` — так модуль сам разберётся с переводами строк
внутри полей. Кодировку, как всегда, указывают явно.

In [ ]:
people = DEMO / "люди.csv"
people.write_text(
    'фамилия,город,возраст\nИванов,"Москва, Россия",30\nПетрова,Казань,25\n',
    encoding="utf-8",
)

with people.open(encoding="utf-8", newline="") as file:
    for row in csv.reader(file):
        print(row)

Первая строка — заголовок, и `reader` не выделяет её никак. Удобнее `DictReader`:
он берёт заголовок и отдаёт словари.

In [ ]:
with people.open(encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

rows

### Всё приходит строками

CSV не хранит типы. Возраст выше — строка `"30"`, и арифметика с ней даст не то,
что ожидается:

In [ ]:
rows[0]["возраст"] + rows[1]["возраст"]

Поэтому сразу после чтения значения приводят к нужным типам:

In [ ]:
ages = [int(row["возраст"]) for row in rows]

sum(ages) / len(ages)

### Запись

`DictWriter` требует список полей: он же задаёт порядок столбцов.

In [ ]:
out = DEMO / "выгрузка.csv"

with out.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["фамилия", "возраст"])
    writer.writeheader()
    writer.writerow({"фамилия": "Иванов", "возраст": 30})

print(out.read_text(encoding="utf-8"))

> **Типичная ошибка.** Файл открыт без `newline=""`. В Windows между строками
> появится лишняя пустая строка, и таблица «поедет». Ошибки при этом не будет —
> проблему заметят только по испорченной выгрузке.

## Даты и время

Три основных типа: `date` — календарная дата, `datetime` — дата со временем,
`timedelta` — промежуток.

In [ ]:
from datetime import UTC, date, datetime, timedelta

release = date(2026, 9, 1)
release, release.year, release.weekday()

`weekday` нумерует дни с нуля: понедельник — 0, воскресенье — 6.

### Арифметика

Разность дат даёт `timedelta`, сложение даты с промежутком — новую дату:

In [ ]:
exam = date(2026, 12, 20)
left = exam - release

left, left.days, release + timedelta(days=7)

### Наивное и осведомлённое время

Это главная развилка темы. `datetime` бывает двух видов:

* **наивный** — просто числа, без указания часового пояса. Непонятно, чьи это
  девять утра;
* **осведомлённый** — с привязкой к поясу.

`now()` без аргументов даёт наивное локальное время, `now(UTC)` — осведомлённое.
Константа `UTC` появилась в версии 3.11; в коде постарше то же самое пишут как
`timezone.utc`.

In [ ]:
naive = datetime.now()
aware = datetime.now(UTC)

naive.tzinfo, aware.tzinfo

Смешивать их нельзя — сравнение и вычитание завершаются ошибкой:

In [ ]:
try:
    print(naive < aware)
except TypeError as error:
    print(type(error).__name__, error)

> **Типичная ошибка.** Данные из разных источников приходят в разных видах —
> одно поле наивное, другое осведомлённое, — и программа падает на первом же
> сравнении, зачастую в проде и через месяц после написания. Правило простое:
> внутри программы держат осведомлённое время в UTC, а в местное переводят
> только при показе человеку.

### Обмен: строки

Для записи в JSON или CSV дату превращают в строку стандартного формата ISO
8601, а при чтении разбирают обратно:

In [ ]:
moment = datetime(2026, 9, 1, 10, 30, tzinfo=UTC)
text = moment.isoformat()

text, datetime.fromisoformat(text) == moment

Если формат нестандартный, его описывают явно:

In [ ]:
datetime.strptime("01.09.2026 10:30", "%d.%m.%Y %H:%M")

Обратная операция — `strftime`:

In [ ]:
moment.strftime("%d.%m.%Y"), moment.strftime("%H:%M %Z")

## Сквозной пример

Соберём всё вместе: прочитать таблицу с датами, отобрать нужные строки,
преобразовать типы и выгрузить результат в JSON.

In [ ]:
orders = DEMO / "заказы.csv"
orders.write_text(
    "номер,создан,сумма\n1,2026-09-01,1500\n2,2026-09-15,700\n3,2026-10-02,2300\n",
    encoding="utf-8",
)

with orders.open(encoding="utf-8", newline="") as file:
    raw_rows = list(csv.DictReader(file))

raw_rows

Разбор: строки превращаем в даты и числа сразу, одним проходом.

In [ ]:
parsed = [
    {
        "номер": int(row["номер"]),
        "создан": date.fromisoformat(row["создан"]),
        "сумма": int(row["сумма"]),
    }
    for row in raw_rows
]

parsed[0]

Отбор — обычное включение по условию на дату:

In [ ]:
september = [row for row in parsed if row["создан"].month == 9]

len(september), sum(row["сумма"] for row in september)

Выгрузка. Дату JSON не умеет, поэтому переводим её в строку — иначе получим
`TypeError`, который разбирали выше.

In [ ]:
result = DEMO / "сентябрь.json"

with result.open("w", encoding="utf-8") as file:
    json.dump(
        [{**row, "создан": row["создан"].isoformat()} for row in september],
        file,
        ensure_ascii=False,
        indent=2,
    )

print(result.read_text(encoding="utf-8"))

## Итого

| Задача | Решение | Чего не делать |
|---|---|---|
| путь к файлу | `Path`, оператор `/` | склеивать строки через `+` |
| путь рядом с кодом | `Path(__file__).parent` | относительный путь от `cwd` |
| чтение текста | `read_text(encoding="utf-8")` | полагаться на кодировку системы |
| строка JSON | `loads` / `dumps` | путать с `load` / `dump` для файлов |
| кириллица в JSON | `ensure_ascii=False` | оставлять `\uXXXX` в файле для людей |
| дата в JSON | `isoformat()` при записи | передавать `date` напрямую |
| таблица | `csv.DictReader` | `split(",")` |
| типы из CSV | приводить сразу после чтения | считать, что пришли числа |
| время внутри программы | осведомлённое, в UTC | смешивать с наивным |

Общее правило для всей лекции: данные из внешнего мира **не обязаны** быть
корректными. Файла может не быть, JSON может оказаться битым, в числовом столбце
может лежать пустая строка. Разбор чужих данных всегда пишут с этой мыслью.

In [ ]:
# уборка: удаляем учебные файлы
import shutil

shutil.rmtree(DEMO)
DEMO.exists()